# Bodrum Hotel & Destination Intelligence
## 03 - Data Cleaning

Bu notebook, `02_data_audit.ipynb` bulgularına dayanarak ana otel tablosunu kayıpsız ve açıklanabilir biçimde temizler.

Temel ilkeler:

- Ham dosya değiştirilmez.
- Eksik değerler tahmin edilmez veya doldurulmaz.
- Google müşteri puanı ile resmî yıldız sınıfı birbirinden ayrı tutulur.
- Fiyat alanı yalnızca tarihli bir arama snapshot'ı olarak korunur.
- Her dönüşüm ve doğrulama kontrolü ayrı rapora yazılır.


### 1. Kurulum ve proje yolları

Notebook ister proje kökünden ister `notebooks/` klasöründen çalıştırılsın aynı dosyaları bulur. Temizleme mantığı tekrar kullanım için `src/bodrum_intelligence/cleaning.py` modülünde tutulur.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from bodrum_intelligence.cleaning import clean_hotels, load_raw_hotels, save_cleaning_outputs

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "bodrum_hotels_master_2026-08-24.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

assert RAW_PATH.exists(), f'Ham veri bulunamadı: {RAW_PATH}'

### 2. Ham verinin güvenli yüklenmesi

Telefon ve kimlik alanları baştaki `+` işareti veya olası sıfırlar kaybolmasın diye metin olarak okunur. Bu aşamada veri üzerinde değişiklik yapılmaz.


In [2]:
raw_df = load_raw_hotels(RAW_PATH)

raw_summary = pd.DataFrame(
    {
        "metric": ["row_count", "column_count", "missing_cells", "duplicate_rows"],
        "value": [
            len(raw_df),
            raw_df.shape[1],
            int(raw_df.isna().sum().sum()),
            int(raw_df.duplicated().sum()),
        ],
    }
)
display(raw_summary)

,metric,value
0,row_count,192
1,column_count,19
2,missing_cells,435
3,duplicate_rows,0


### 3. Temizleme dönüşümleri

Uygulanan işlemler yalnızca baş/son boşluklarını temizleme, boş metinleri standart eksik değere çevirme, güvenli nullable veri tipleri oluşturma ve tarihi ISO biçimine getirmedir. Satır silme, aykırı değer baskılama ve imputasyon yapılmaz.


In [3]:
result = clean_hotels(raw_df)
df = result.hotels

display(result.transformation_log)
display(df.dtypes.rename("dtype").astype(str).to_frame())

,step,column,affected_rows,note
0,trim_whitespace,hotel_id,0,Baş/son boşlukları kaldırıldı.
1,empty_string_to_null,hotel_id,0,Boş metinler eksik değer olarak işaretlendi.
2,trim_whitespace,place_id,0,Baş/son boşlukları kaldırıldı.
3,empty_string_to_null,place_id,0,Boş metinler eksik değer olarak işaretlendi.
4,trim_whitespace,hotel_name,0,Baş/son boşlukları kaldırıldı.
5,empty_string_to_null,hotel_name,0,Boş metinler eksik değer olarak işaretlendi.
6,trim_whitespace,area,0,Baş/son boşlukları kaldırıldı.
7,empty_string_to_null,area,0,Boş metinler eksik değer olarak işaretlendi.
8,trim_whitespace,district,0,Baş/son boşlukları kaldırıldı.
9,empty_string_to_null,district,0,Boş metinler eksik değer olarak işaretlendi.


,dtype
hotel_id,string
place_id,string
hotel_name,string
area,string
district,string
province,string
country,string
property_category,string
official_star_rating,Float64
google_rating,Float64


### 4. Eksik değerlerin korunması

Özellikle `official_star_rating`, `business_status` ve fiyat snapshot'ındaki eksiklikler bilgi yokluğunu temsil eder. Bu değerler başka kolonlardan türetilmez.


In [4]:
important_missing = [
    "official_star_rating",
    "business_status",
    "search_price_usd_snapshot",
    "phone",
]

missing_comparison = pd.DataFrame(
    {
        "column": important_missing,
        "raw_missing": [int(raw_df[column].isna().sum()) for column in important_missing],
        "clean_missing": [int(df[column].isna().sum()) for column in important_missing],
    }
)
missing_comparison["difference"] = missing_comparison["clean_missing"] - missing_comparison["raw_missing"]
display(missing_comparison)

,column,raw_missing,clean_missing,difference
0,official_star_rating,192,192,0
1,business_status,191,191,0
2,search_price_usd_snapshot,24,24,0
3,phone,4,4,0


### 5. Doğrulama kontrolleri

Temiz veri; satır sayısı, anahtar bütünlüğü ve alanların geçerli aralıkları açısından kontrol edilir. Herhangi bir `FAIL`, sonraki analize geçmeden önce incelenmelidir.


In [5]:
display(result.validation_report)

failed_checks = result.validation_report.loc[result.validation_report["status"].eq("FAIL")]
assert failed_checks.empty, f'Başarısız temizleme kontrolleri:\n{failed_checks.to_string(index=False)}'

,check,status,issue_count,detail
0,row_count_preserved,PASS,0,Temizleme satır eklememeli veya silmemelidir.
1,required_ids_present,PASS,0,hotel_id ve place_id zorunludur.
2,required_ids_unique,PASS,0,hotel_id/place_id çifti benzersiz olmalıdır.
3,google_rating_in_range,PASS,0,Google müşteri puanı 0-5 aralığında olmalıdır.
4,google_rating_scale_is_five,PASS,0,Mevcut puan ölçeği 5 olmalıdır.
5,review_count_non_negative,PASS,0,Yorum sayısı negatif olamaz.
6,snapshot_price_positive,PASS,0,Dolu fiyat snapshot değeri pozitif olmalıdır.
7,official_star_in_range,PASS,0,Dolu resmî yıldız 1-5 aralığında olmalıdır.
8,area_hotel_count_matches,PASS,0,Bölge otel sayısı ana tablodan türetilmelidir.


### 6. İşlenmiş verinin ve raporların kaydedilmesi

Temiz tablo `data/processed/` altına; dönüşüm günlüğü ve doğrulama raporu `reports/` altına yazılır. `data/raw/` içeriğine yazılmaz.


In [6]:
output_paths = save_cleaning_outputs(result, PROCESSED_DIR, REPORTS_DIR)
pd.DataFrame(
    [{"output": name, "path": str(path.relative_to(PROJECT_ROOT))} for name, path in output_paths.items()]
)

,output,path
0,clean_hotels,data/processed/hotels_clean.csv
1,transformation_log,reports/cleaning_transformation_log.csv
2,validation_report,reports/cleaning_validation_report.csv


### 7. Sonuç

- **192 kayıt korunmuştur;** satır eklenmemiş veya silinmemiştir.
- Tek çalışma tablosu `df` kullanılmış ve bölge otel sayısı `area_hotel_count` olarak bu tablodan türetilmiştir.
- Otel ve Google Place kimlikleri benzersiz kalmıştır.
- Telefon alanı metin, sayısal alanlar nullable sayısal tip olarak standardize edilmiştir.
- Resmî yıldız bilgisi uydurulmamış; mevcut eksiklik aynen korunmuştur.
- Fiyat snapshot'ı eksikleri doldurulmamış ve aykırı görünen değerler silinmemiştir.
- Temiz veri ve dönüşüm kanıtları ham kaynaktan ayrı kaydedilmiştir.

Bir sonraki aşamada `04_feature_engineering.ipynb`, yalnızca araştırma sorularına hizmet eden ve veri kapsamıyla desteklenen özellikleri üretmelidir.
